# RT-DETR R50-VD (COCO) — DIMER real-time object detection tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/rtdetr-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/rtdetr-detection-pipeline/blob/main/tutorials/rtdetr_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-PekingU%2Frtdetr__r50vd-ffcc4d?style=flat)](https://huggingface.co/PekingU/rtdetr_r50vd) [![Upstream](https://img.shields.io/badge/Upstream-lyuwenyu%2FRT--DETR-181717?style=flat&logo=github&logoColor=white)](https://github.com/lyuwenyu/RT-DETR) [![arXiv](https://img.shields.io/badge/arXiv-2304.08069-b31b1b.svg)](https://arxiv.org/abs/2304.08069)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** object detection over the 80 COCO classes on one image (score-ordered xyxy boxes with a per-class sigmoid score under a caller-owned threshold) using the pinned `PekingU/rtdetr_r50vd` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/rtdetr_detection_pipeline/pipeline.py` at revision `fc1a3785500f`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `df939e661d8c52e80608d1ec566561aabd25a4e7` (~172 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the RT-DETR model reads one image resized to 640×640 (aspect ratio not preserved), runs a ResNet-50-vd backbone, a one-layer hybrid encoder over three feature scales and a 6-layer transformer decoder with 300 learned object queries, and emits, per query, a box and an **independent sigmoid score per COCO class** (the model is trained with a focal loss, so scores are not a softmax over classes); the processor keeps the top (query, class) pairs above the threshold and maps their boxes back to input pixels. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and image-processor configuration, and the carried module adds snapshot verification, the input contract, a fixed output contract and the `box_iou`, `validate_inputs` and `evaluation_report` helpers. The default sample is a street-like scene drawn in code (a stop sign, a traffic light, an analogue clock and a sports ball) whose drawn boxes serve as references; the resulting per-object `box_iou` values are demonstration (plumbing) evidence for one image, not a detection benchmark — and one of the four drawn objects is not detected at all, which the notebook records rather than hides.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic scene with reference boxes per COCO class (or upload your own image) and validate it into an input manifest, run the supported task, read the 80 labels, the sigmoid scores and the caller-owned threshold correctly, exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with per-object `box_iou` only when reference boxes exist and `not-measurable` otherwise, and export machine-readable detections plus an annotated image and provenance.

**This notebook does not demonstrate:** classes outside the 80 COCO categories (a closed vocabulary: use an open-vocabulary detector such as OWLv2 or Grounding DINO for text prompts), instance segmentation, tracking, batched or video inference, COCO mean-average-precision evaluation (which needs a labelled image set; only per-object `box_iou` against drawn references is computed here), the upstream latency claims (108 FPS on a T4 with TensorRT; this notebook measures CPU wall time only), or any training. The model was trained on COCO 2017 photographs; drawn icons, documents, medical or aerial imagery and non-COCO objects are outside what this notebook measures, and a scene with no objects still yields boxes.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 4.8 s to load and 0.25–0.34 s per `detect` on the 640×480 synthetic scene in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 172 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union measures; the difference between a sigmoid per-class score and a softmax over classes.
- **Data:** the default sample is a deterministic 640×480 scene drawn in code (sky, ground, a road edge, a red octagonal stop sign with the word STOP, a three-lamp traffic light, a white analogue clock with numerals and hands, and an orange sports ball with seams), so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar), ideally a photograph of everyday scenes, any colour mode, sides between 16 and 4096 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `PekingU/rtdetr_r50vd` snapshot (~172 MB in total) at revision `df939e661d8c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'rtdetr-detection-pipeline',
    'repository_revision': 'fc1a3785500f57e3bfeed74f71cf4ae9dd384727',
    'embedded_module': 'src/rtdetr_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/rtdetr_detection_pipeline/pipeline.py'],
    'module_sha256': 'be2bd1b1240c54baa089f4d5f4c97166b8fed0d9a61c890c7a2712d1616845d7',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/rtdetr_detection_pipeline/` @ `fc1a3785500f`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/rtdetr_detection_pipeline/pipeline.py`

In [ ]:
"""Real-time object detection with the pinned ``PekingU/rtdetr_r50vd`` checkpoint (RT-DETR, COCO classes).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the RT-DETR architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "PekingU/rtdetr_r50vd"
MODEL_REVISION = "df939e661d8c52e80608d1ec566561aabd25a4e7"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "rtdetr-r50vd"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The 80 COCO 2017 classes the checkpoint was trained on, in config.json id2label order (the
# upstream spelling: "motorbike", "aeroplane", "sofa", "pottedplant", "tvmonitor", ...).
LABELS = (
    "person",
    "bicycle",
    "car",
    "motorbike",
    "aeroplane",
    "bus",
    "train",
    "truck",
    "boat",
    "traffic light",
    "fire hydrant",
    "stop sign",
    "parking meter",
    "bench",
    "bird",
    "cat",
    "dog",
    "horse",
    "sheep",
    "cow",
    "elephant",
    "bear",
    "zebra",
    "giraffe",
    "backpack",
    "umbrella",
    "handbag",
    "tie",
    "suitcase",
    "frisbee",
    "skis",
    "snowboard",
    "sports ball",
    "kite",
    "baseball bat",
    "baseball glove",
    "skateboard",
    "surfboard",
    "tennis racket",
    "bottle",
    "wine glass",
    "cup",
    "fork",
    "knife",
    "spoon",
    "bowl",
    "banana",
    "apple",
    "sandwich",
    "orange",
    "broccoli",
    "carrot",
    "hot dog",
    "pizza",
    "donut",
    "cake",
    "chair",
    "sofa",
    "pottedplant",
    "bed",
    "diningtable",
    "toilet",
    "tvmonitor",
    "laptop",
    "mouse",
    "remote",
    "keyboard",
    "cell phone",
    "microwave",
    "oven",
    "toaster",
    "sink",
    "refrigerator",
    "book",
    "clock",
    "vase",
    "scissors",
    "teddy bear",
    "hair drier",
    "toothbrush",
)
# Detection threshold: the value the pinned README's transformers example passes to
# post_process_object_detection (threshold=0.3). RT-DETR is trained with a focal (sigmoid) loss, so
# each score is an independent per-class sigmoid, not a softmax over classes; the value was not
# calibrated for any deployment and the deployment owns tuning it on labelled images.
DETECTION_THRESHOLD = 0.3
# The decoder emits exactly num_queries proposals (config.json) and the processor keeps at most that
# many (query, class) pairs, so no image can yield more than this many boxes.
MAX_DETECTIONS = 300
# Input ceilings. The processor resizes every image to 640x640 (preprocessor_config.json `size`,
# aspect ratio not preserved) and rescales to [0, 1] without mean/std normalisation, so image cost
# is bounded whatever the caller sends; the side ceiling only guards memory during decoding.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for any caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"threshold must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB): a photograph or a rendered scene",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB; the processor resizes to 640x640 without preserving the aspect ratio "
        "and rescales to [0, 1] (no mean/std normalisation); returned boxes are mapped back to input pixels"
    ),
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``detect`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[Sequence[float]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (a mapping label -> xyxy reference boxes of the objects in the image)
    the report carries one ``box_iou`` entry per reference — the best-overlapping detection **of the
    same label** — as sample-sanity geometry evidence; without them the verdict is ``not-measurable``
    and the report says what labelled data would make the task measurable.
    """
    detections = list(result["detections"])
    base = {
        "task": "object detection over the 80 COCO classes on one image",
        "decision_rule": (
            "a (query, class) pair survives when its sigmoid class score reaches the threshold; the score "
            "is an independent per-class sigmoid under the model's own focal-loss head, not a calibrated "
            "probability for the deployment's images, and one query can surface under several classes"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth object boxes were supplied for the evaluated image",
            "needs": (
                "labelled boxes per class on your own images, scored per object with box_iou and aggregated "
                "into COCO-style mean average precision (AP@[.50:.95], AP50) at stated IoU thresholds; no "
                "such labelled set ships with this repository"
            ),
        }
    metrics = []
    for label, boxes in ground_truth_boxes.items():
        if label not in LABELS:
            raise ValueError(f"unknown reference label {label!r}; expected one of the 80 COCO LABELS")
        same_label = [det for det in detections if det["label"] == label]
        for index, box in enumerate(boxes):
            ious = [box_iou(det["box"], box) for det in same_label]
            best = max(range(len(ious)), key=ious.__getitem__) if ious else None
            metrics.append(
                {
                    "id": "box_iou",
                    "reference": f"{label}-{index}",
                    "value": ious[best] if best is not None else 0.0,
                    "matched_score": same_label[best]["score"] if best is not None else None,
                    "n_detected_same_label": len(same_label),
                    "estimation": "one reference box per object on a single image, no dispersion estimate",
                }
            )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial image; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled image set from the deployment domain (cameras, scenes, object classes) for any "
            "COCO-style mean-average-precision or precision/recall claim"
        ),
    }


@dataclass
class RTDetrDetectionPipeline:
    """COCO-class object detection over the pinned RT-DETR (ResNet-50-vd) checkpoint."""

    _runner: Callable[[Image.Image, float], list[dict[str, Any]]]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> RTDetrDetectionPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoImageProcessor, RTDetrForObjectDetection

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        # config.json already says use_pretrained_backbone=false and use_timm_backbone=false (the
        # ResNet-50-vd backbone is the in-library RTDetrResNet); it is passed explicitly anyway so a
        # future config change cannot re-introduce a construction-time weight download.
        model = RTDetrForObjectDetection.from_pretrained(
            source,
            revision=MODEL_REVISION,
            trust_remote_code=False,
            use_pretrained_backbone=False,
            **kwargs,
        )
        model = model.to(resolved_device).eval()
        id2label = {int(k): v for k, v in model.config.id2label.items()}

        def runner(image: Image.Image, threshold: float) -> list[dict]:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs)
            # use_focal_loss=True: per-class sigmoid scores, top num_queries (query, class) pairs,
            # then the threshold — the pinned processor's default for this checkpoint.
            result = processor.post_process_object_detection(
                outputs, threshold=threshold, target_sizes=[image.size[::-1]]
            )[0]
            return [
                {
                    "box": [float(v) for v in box.tolist()],
                    "label": id2label[int(label)],
                    "score": float(score),
                }
                for box, label, score in zip(result["boxes"], result["labels"], result["scores"], strict=True)
            ]

        return cls(runner, resolved_device)

    def detect(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Detect COCO-class objects on one image; boxes are xyxy pixel coordinates in the input image."""
        rgb, checked = _check_inputs(image, threshold)
        detections = self._runner(rgb, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > num_queries {MAX_DETECTIONS}"
            )
        for det in detections:
            if set(det) != {"box", "label", "score"} or len(det["box"]) != 4 or det["label"] not in LABELS:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `df939e661d8c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `RTDetrDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "rtdetr-r50vd",
  "modelId": "PekingU/rtdetr_r50vd",
  "revision": "df939e661d8c52e80608d1ec566561aabd25a4e7",
  "files": [
    {
      "path": "README.md",
      "bytes": 9053,
      "sha256": "4a0c10ddd0dbf6a2c6815cfc1613a5d74f3b0e7c8c77f0252212d3e5365e98cb"
    },
    {
      "path": "config.json",
      "bytes": 5113,
      "sha256": "2ed2a305c51eef46715eb755a02b2a266ecfb752936cc9574bb5714601c2742d"
    },
    {
      "path": "model.safetensors",
      "bytes": 172175856,
      "sha256": "5263d5521eff3e356f6cd8a371fd5dfb891725beda5f713674f79669115cdc64"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 841,
      "sha256": "ffb4b9461a1dad746be8f0f9c8330ed7743a1ba5fba4f75c232cd281b3d4c64a"
    }
  ],
  "totalBytes": 172190863
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = RTDetrDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scene or optional BYOD

The default sample is **synthetic** and carries its own reference boxes: a 640×480 scene drawn with Pillow — sky and ground, a red octagonal **stop sign** with the word STOP on a post, a black three-lamp **traffic light**, a white analogue **clock** with numerals and hands, and an orange **sports ball** with seams — the same scene the repository's smoke run used. The drawn boxes, keyed by their COCO label, are the references for the per-object `box_iou` sanity check later; they are not a labelled dataset, so nothing here is a mean-average-precision measurement, and drawn icons are not the photographs the model was trained on. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one image — no reference boxes exist for it, so the evaluation report will be `not-measurable`.

The detection threshold is a **caller-owned request parameter**, not a pipeline constant: a (query, class) pair survives when its sigmoid class score reaches it. The package default (`DETECTION_THRESHOLD = 0.3`) is the value the pinned README's transformers example passes, not a calibration; it is exposed here as a form parameter and passed explicitly on every call. Nothing is validated in this cell — the next section hands the image and the threshold to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the threshold, and the reference boxes per label.

In [ ]:
import hashlib
import io
import math

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
threshold = 0.3  # @param {type:"number"}


def synthetic_scene(width=640, height=480):
    """Sky/ground scene with a stop sign, a traffic light, an analogue clock and a sports ball; returns image + label->boxes."""
    img = Image.new('RGB', (width, height), (135, 190, 235))
    d = ImageDraw.Draw(img)
    d.rectangle([0, 330, width, height], fill=(96, 128, 72))
    d.rectangle([0, 300, width, 330], fill=(110, 110, 110))
    refs = {}
    cx, cy, r = 110, 150, 62
    pts = [(cx + r * math.cos(math.pi / 8 + k * math.pi / 4), cy + r * math.sin(math.pi / 8 + k * math.pi / 4)) for k in range(8)]
    d.rectangle([cx - 5, cy, cx + 5, 330], fill=(90, 90, 90))
    d.polygon(pts, fill=(200, 20, 30), outline=(255, 255, 255))
    f = ImageFont.load_default(size=30)
    d.text((cx - d.textlength('STOP', font=f) / 2, cy - 17), 'STOP', fill='white', font=f)
    refs['stop sign'] = [[cx - r, cy - r, cx + r, cy + r]]
    x0, y0 = 270, 60
    d.rectangle([x0 + 22, y0 + 150, x0 + 30, 330], fill=(70, 70, 70))
    d.rectangle([x0, y0, x0 + 52, y0 + 150], fill=(25, 25, 25), outline=(60, 60, 60))
    for k, col in enumerate([(230, 30, 30), (240, 200, 30), (40, 200, 60)]):
        d.ellipse([x0 + 8, y0 + 8 + k * 47, x0 + 44, y0 + 44 + k * 47], fill=col)
    refs['traffic light'] = [[x0, y0, x0 + 52, y0 + 150]]
    cx, cy, r = 480, 140, 70
    d.rectangle([cx - 6, cy, cx + 6, 330], fill=(120, 80, 40))
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 250, 245), outline=(20, 20, 20), width=5)
    f2 = ImageFont.load_default(size=16)
    for h in range(1, 13):
        a = math.radians(h * 30 - 90)
        d.text((cx + (r - 18) * math.cos(a) - 5, cy + (r - 18) * math.sin(a) - 8), str(h), fill='black', font=f2)
    d.line([(cx, cy), (cx + 0.5 * r * math.cos(math.radians(-60)), cy + 0.5 * r * math.sin(math.radians(-60)))], fill='black', width=5)
    d.line([(cx, cy), (cx + 0.8 * r * math.cos(math.radians(30)), cy + 0.8 * r * math.sin(math.radians(30)))], fill='black', width=3)
    refs['clock'] = [[cx - r, cy - r, cx + r, cy + r]]
    cx, cy, r = 330, 400, 45
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(235, 120, 30), outline=(40, 20, 10), width=3)
    d.line([(cx - r, cy), (cx + r, cy)], fill=(40, 20, 10), width=3)
    d.line([(cx, cy - r), (cx, cy + r)], fill=(40, 20, 10), width=3)
    d.arc([cx - r * 1.6, cy - r, cx - r * 0.2, cy + r], 300, 60, fill=(40, 20, 10), width=3)
    d.arc([cx + r * 0.2, cy - r, cx + r * 1.6, cy + r], 120, 240, fill=(40, 20, 10), width=3)
    refs['sports ball'] = [[cx - r, cy - r, cx + r, cy + r]]
    return img, {label: [[float(v) for v in box] for box in boxes] for label, boxes in refs.items()}


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    drawn_boxes = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic scene: no randomness, so no seed is needed and the digest is stable per Pillow build.
    image, drawn_boxes = synthetic_scene()
    image_name = 'synthetic_scene_640x480.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'threshold': threshold, 'reference_boxes': drawn_boxes})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `detect` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px and a threshold in `[0, 1]` — and returns an **input manifest** naming the schema (including the 80 labels and the 300-query ceiling on detections), the input's observed mode and size, the threshold, and the verdict. The manifest is written to `outputs/rtdetr_detection_input_manifest.json`. To show what rejection looks like, the cell also validates a threshold outside `[0, 1]` and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and resized to 640×640 by the processor (the aspect ratio is not preserved — a tall or wide image is squashed); boxes are mapped back to input pixels, and nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_DETECTIONS': MAX_DETECTIONS, 'n_labels': len(LABELS), 'DETECTION_THRESHOLD': DETECTION_THRESHOLD}})
print({'LABELS': list(LABELS)})
input_manifest = validate_inputs(image, threshold=threshold, names=[image_name])
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(image, threshold=1.5)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'out-of-range-threshold-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/rtdetr_detection_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in input_manifest.items() if k != 'schema'}, indent=2))

## 6. Detect and read the scores correctly

`detect` returns a dict with `detections` — a list of `{box, label, score}` **ordered by descending score**, `box` in xyxy pixel coordinates of the input, `label` one of the 80 COCO classes — plus the threshold used, `width`, `height` and the model identity. At most 300 boxes can ever be returned (the decoder has 300 queries and the processor keeps at most that many (query, class) pairs, so one query can surface twice under two labels). Each `score` is the **per-class sigmoid under the model's own focal-loss head, not a calibrated estimate for your images**: it was never fitted to the frequency with which a box is a real object on your data, and the scores of different classes for one query do not sum to one. The threshold you passed is the only decision rule; the pipeline ships 0.3 as a default (the README example's value), not as a calibration, and the caller owns it per deployment. Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move scores in the third or fourth decimal place. As recorded in the model card, the repository's CPU smoke on this same scene returned exactly three boxes — `stop sign` 0.977, `clock` 0.965, `traffic light` 0.931 — and **no `sports ball`** even at threshold 0.1; the same three boxes appeared at 0.1, 0.5 and 0.9. That is one observation on drawn icons, not a calibration point.

In [ ]:
import time

t0 = time.time()
result = pipe.detect(image, threshold=threshold)
elapsed = time.time() - t0
print({'n_detections': len(result['detections']), 'threshold': result['threshold'], 'device': pipe.device, 'seconds': round(elapsed, 2)})
for rank, det in enumerate(result['detections'], start=1):
    print(f"{rank:>3}. score {det['score']:.4f}  label {det['label']!r:16}  box {[round(v, 1) for v in det['box']]}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No detection metric is reported by default: COCO mean average precision needs a labelled image set, and this repository ships none. The repository's only metric helper is `box_iou(a, b)` (intersection-over-union of two xyxy boxes), the building block a caller would use to compute mAP on their own labelled images; when reference boxes are supplied, keyed by COCO label, the report carries one `box_iou` entry per reference — matched only against detections **of the same label**, with the matched detection's score and how many same-label detections existed — with the verdict `sample-sanity`. A reference with no same-label detection scores 0.0 and `n_detected_same_label` 0: that is what a miss looks like, and on this scene the sports ball is one. On the synthetic path those references are icons **you drew yourself**, so a high IoU proves only that the input contract, forward pass and coordinate mapping round-trip. On BYOD no reference exists, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/rtdetr_detection_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, drawn_boxes, sample_kind=sample_kind)
with open('outputs/rtdetr_detection_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k != 'metrics'}, indent=2))
for metric in report['metrics']:
    matched = 'no same-label detection' if metric['matched_score'] is None else f"matched score {metric['matched_score']:.3f}"
    print(f"{metric['reference']:18} iou {metric['value']:.3f}  ({matched}, same-label detections {metric['n_detected_same_label']})")
if report['verdict'] == 'not-measurable':
    print('No reference boxes exist for this input, so box_iou is not computed; inspect the annotated PNG instead.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (score-ordered detections with boxes and labels, the threshold), the evaluation report, the input manifest, the sample identity, digest and reference boxes, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The detections are also written as CSV with explicit `image`, `rank`, `label`, `score`, `x0`, `y0`, `x1`, `y1` columns so score ordering survives downstream use, and an annotated PNG draws every returned box in green with its label and score, and every reference box in red, for visual inspection (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import csv

annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for label, boxes in (drawn_boxes or {}).items():
    for box in boxes:
        draw.rectangle(box, outline=(220, 30, 30), width=2)
for det in result['detections']:
    draw.rectangle(det['box'], outline=(0, 160, 0), width=3)
    draw.text((det['box'][0] + 4, det['box'][1] + 4), f"{det['label']} {det['score']:.3f}", fill=(0, 160, 0))
annotated.save('outputs/rtdetr_detection_annotated.png')
payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'reference_boxes': drawn_boxes},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/rtdetr_detection_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/rtdetr_detection_detections.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1'])
    for rank, det in enumerate(result['detections'], start=1):
        writer.writerow([image_name, rank, det['label'], f"{det['score']:.6f}", *[f"{v:.2f}" for v in det['box']]])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The boxes locate regions the model classifies as one of the 80 COCO classes; the vocabulary is closed, the sigmoid score is not calibrated for your images, and the threshold is a request parameter you own (the default is the README example's value, not a tuned operating point). On the synthetic scene the per-object `box_iou` values in the evaluation report compare detections to icons you drew yourself and the verdict is `sample-sanity`, which proves only that the input contract, forward pass and coordinate mapping work — and the drawn sports ball, which the model does not find at any threshold, shows that a drawn icon is not a photograph; they say nothing about real scenes, small or occluded objects, crowded images, unusual viewpoints or non-COCO objects, and a BYOD result is a single-image observation with the verdict `not-measurable`. **The model emits boxes for any image**: the repository's smoke run fed it a blank 4096×4096 image and got one `train` at 0.33, and a uniform-noise image one `cat` at 0.32, so an empty scene at the default threshold produces a confident nonsense box rather than an empty result. Everything is resized to 640×640, so tiny objects and extreme aspect ratios suffer. The pipeline provides no open-vocabulary prompting, no segmentation, no tracking, no mAP evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** lower `threshold` to 0.05 and look for the sports ball (the smoke run found nothing at 0.1); redraw the ball with a soccer-ball pattern or a photograph-like shading and see whether it appears; raise `threshold` to 0.9 and check the three boxes survive (they did); enable `USE_BYOD` with a street photograph, hand-label a few objects by COCO class and pass them to `evaluation_report` to see the verdict switch to `sample-sanity` — the first step towards a real precision/recall number.

## References

- Repository README: https://github.com/kurtvalcorza/rtdetr-detection-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/rtdetr-detection-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/rtdetr-detection-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/PekingU/rtdetr_r50vd
- Upstream code: https://github.com/lyuwenyu/RT-DETR
- DETRs Beat YOLOs on Real-time Object Detection (Zhao et al., 2023): https://arxiv.org/abs/2304.08069
- Microsoft COCO: Common Objects in Context (Lin et al., 2014): https://arxiv.org/abs/1405.0312